# Knowledge Base and Ingest for RAG

RAG quality has a **ceiling** set before the first user query. If the knowledge base is stale, noisy, missing key documents, or chunked poorly, no prompt tweak or larger LLM will reliably fix retrieval. **Ingest** is the offline pipeline that turns raw sources into searchable, versioned chunks in your vector store.

A **knowledge base (KB)** is not just a folder of PDFs — it is a **managed product**: scoped sources, normalization rules, metadata schema, embedding version, access controls, and a refresh cadence. **Ingest** is the job that materializes that product into vectors.

This notebook covers corpus design and governance, the full ingest pipeline (load → clean → chunk → enrich → embed → upsert), metadata and chunk identity, incremental updates, data quality gates, PII/ACL concerns, observability, and wiring ingest output to the online RAG path from notebook **02**. Chunking *strategies* (size, overlap, structure-aware splits) are detailed in **05.04 Document Chunking** — here we focus on **KB lifecycle** and ingest operations.

Prerequisites: **03. RAG Architecture Patterns**, **02. Naive RAG Pipeline**, **05. Embeddings & Vector DBs** (**03** APIs, **04** chunking, **06** Chroma). Next: **05. Prompting and Context Injection for RAG**.

In [ ]:
import hashlib
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone

import chromadb
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. Why the Knowledge Base Comes First

Notebook **01** showed that LLMs lack private, fresh facts. Notebook **02** assumed chunks **already exist**. This notebook answers: **where do those chunks come from**, and how do you keep them correct over time?

### 1.1 The Quality Ceiling

```
Raw sources (messy)  ──►  Ingest (structured)  ──►  Index  ──►  RAG answers
         ▲                        │
         └──── if ingest fails ───┘  retrieval cannot recover
```

| Upstream mistake | Downstream symptom |
|------------------|-------------------|
| Missing doc in corpus | "I don't know" or hallucination |
| Stale vectors | Correct-looking wrong answer |
| Bad chunk boundaries | Right doc, wrong paragraph in top-k |
| No ACL metadata | Leak restricted content |
| Secrets embedded | Irreversible exposure via search |

### 1.2 Ingest vs Online Query

| Phase | When | Cost profile |
|-------|------|--------------|
| **Ingest (offline)** | Batch, on deploy, scheduled | Embed *all* chunks — can be large |
| **Query (online)** | Per user question | Embed *one* query + LLM call |

Optimize ingest for **correctness and idempotency**. Optimize query for **latency**. Do not conflate the two code paths.

---

## 2. Corpus Scope and Governance

Define **in-scope** knowledge explicitly before writing loaders.

### 2.1 Scope Questions

| Question | Example (Notes API project) |
|----------|----------------------------|
| **What sources?** | `README`, OpenAPI spec, `docs/migrations.md`, test guides |
| **What is excluded?** | Slack threads, stale Confluence, `.env`, credentials |
| **Update cadence?** | Re-index on every merge to `main` |
| **Language?** | English only for v1 |
| **Owner?** | Platform team reviews corpus changes |

### 2.2 Curated vs Dump-Everything

| Approach | Result |
|----------|--------|
| **Curated corpus** | Higher precision; easier eval |
| **Dump all files** | Noise, duplicates, conflicting versions |

A smaller, **curated** corpus often beats indexing every PDF in the company. Add sources when eval (**05.09**, **09**) shows coverage gaps — not preemptively.

### 2.3 Source Inventory (Demo Project)

| Source path | Format | Maps to chunks |
|-------------|--------|----------------|
| `docs/api.md` | Markdown | c1 |
| `docs/migrations.md` | Markdown | c2, c5 |
| `docs/security.md` | Markdown | c3 |
| `docs/testing.md` | Markdown | c4 |

---

## 3. The Ingest Pipeline

Every production ingest implements the same stages:

```
Sources ──► Load ──► Clean ──► Chunk ──► Enrich ──► Embed ──► Upsert
                                              │
                                         metadata + stable ids
```

### 3.1 Stage Outputs

| Stage | Input | Output |
|-------|-------|--------|
| **Load** | Files, URLs, tickets | Normalized text + source URI |
| **Clean** | Raw text | Stripped boilerplate, fixed encoding |
| **Chunk** | Long text | Segments with stable ids (see **05.04**) |
| **Enrich** | Chunks | `source`, `section`, `updated_at`, ACL |
| **Embed** | Chunk texts | Vectors (batched API calls) |
| **Upsert** | ids + vectors + metadata | Idempotent write to vector DB |

### 3.2 Idempotency Rule

Running ingest twice on **unchanged** sources should not duplicate vectors. Use **stable chunk ids** and upsert semantics (`add` with same id overwrites in Chroma).

---

## 4. Load — Reading Sources

Match the loader to file format. Treating PDFs as plain text loses structure; treating code as prose breaks function boundaries.

### 4.1 Loaders by Format

| Format | Considerations | Metadata to capture |
|--------|----------------|---------------------|
| **Markdown / RST** | Preserve `#` headings as `section` | `path`, heading trail |
| **HTML docs** | Strip nav, footers, cookie banners | `canonical_url` |
| **OpenAPI JSON** | One chunk per endpoint `description` | `operation_id`, `tag` |
| **PDF** | Watch column order artifacts; OCR noise | `page` numbers |
| **Code** | Split on functions/classes; keep imports context | `filepath`, `symbol` |

### 4.2 Loader Output Contract

Every loader should return a uniform record:

```python
@dataclass
class RawDocument:
    source_path: str
    text: str
    format: str
    loaded_at: str  # ISO timestamp
```

Downstream stages never care whether text came from PDF or Git — only the normalized record.

---

## 5. Clean — Normalization

Cleaning reduces noise that pollutes embeddings.

| Step | Example |
|------|--------|
| **Encoding** | Fix UTF-8 mojibake |
| **Whitespace** | Collapse runs of blank lines |
| **Boilerplate** | Remove "Table of Contents", copyright footers |
| **Secrets scan** | Block lines matching `sk-`, `password=` patterns |
| **De-duplication** | Hash normalized text; skip exact duplicates |

### 5.1 Handoff to Chunking

After clean, text is still **too long** for a single vector. Chunking (**05.04**) splits on token count, recursive separators (`\n\n`, `\n`), or document structure (Markdown headings). Ingest orchestration **calls** the chunker — it does not re-teach chunk algorithms here.

In [ ]:
# Simulated: raw markdown → clean text → logical chunks (pre-chunked for demo)
RAW_DOCS = [
    {
        "path": "docs/api.md",
        "text": "# Notes API\n\nThe Notes API is built with FastAPI and stores notes in memory for demos.\nRoutes: GET/POST /notes.",
    },
    {
        "path": "docs/migrations.md",
        "text": "## Upgrade\nAlembic applies SQLAlchemy schema migrations. Run alembic upgrade head.\n\n## Autogenerate\nAlembic autogenerate compares models to live schema.",
    },
]


def clean_text(text: str) -> str:
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)  # strip heading markers for demo
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def secret_scan(text: str) -> list[str]:
    patterns = [r"sk-[A-Za-z0-9]+", r"password\s*=\s*\S+"]
    hits = []
    for pat in patterns:
        hits.extend(re.findall(pat, text, flags=re.IGNORECASE))
    return hits


for doc in RAW_DOCS:
    cleaned = clean_text(doc["text"])
    flags = secret_scan(cleaned)
    print(f"{doc['path']}: {len(cleaned)} chars, secret_flags={flags or 'none'}")

---

## 6. Metadata Schema Design

Metadata powers **filtering** at query time (**05.07**, **07**) and **audit** after generation.

### 6.1 Recommended Fields

```json
{
  "source": "db-docs",
  "path": "docs/migrations.md",
  "section": "upgrade",
  "updated_at": "2026-03-01T00:00:00Z",
  "product": "notes-api",
  "version": "1.2",
  "acl": "internal"
}
```

| Field | Used for |
|-------|----------|
| `source` | Router / index selection (**03**) |
| `path` | Citations shown to users |
| `section` | Heading breadcrumb in answers |
| `updated_at` | Staleness warnings, re-ingest triggers |
| `acl` | Query-time `where` filters |

### 6.2 Stable Chunk ids

Ids like **c2** let you **upsert** when `docs/migrations.md` changes without orphaning old vectors. Avoid random UUIDs per run unless you also delete stale ids.

In [ ]:
# Full demo corpus with rich metadata (c1–c5)
INGEST_RECORDS = [
    {
        "id": "c1",
        "text": "The Notes API is built with FastAPI and stores notes in memory for demos. Routes live under /notes.",
        "metadata": {
            "source": "api-docs",
            "path": "docs/api.md",
            "section": "overview",
            "version": "1.2",
            "acl": "internal",
        },
    },
    {
        "id": "c2",
        "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling new revisions.",
        "metadata": {
            "source": "db-docs",
            "path": "docs/migrations.md",
            "section": "upgrade",
            "version": "1.2",
            "acl": "internal",
        },
    },
    {
        "id": "c3",
        "text": "JWT bearer tokens authenticate API requests. Clients send Authorization: Bearer token header.",
        "metadata": {
            "source": "security-docs",
            "path": "docs/security.md",
            "section": "auth",
            "version": "1.2",
            "acl": "internal",
        },
    },
    {
        "id": "c4",
        "text": "Pytest fixtures share database setup across tests. Use conftest.py for session-scoped engines.",
        "metadata": {
            "source": "test-docs",
            "path": "docs/testing.md",
            "section": "fixtures",
            "version": "1.2",
            "acl": "internal",
        },
    },
    {
        "id": "c5",
        "text": "Alembic autogenerate compares SQLAlchemy models to the live database schema and drafts revision files.",
        "metadata": {
            "source": "db-docs",
            "path": "docs/migrations.md",
            "section": "autogenerate",
            "version": "1.2",
            "acl": "internal",
        },
    },
]

for r in INGEST_RECORDS:
  meta = r["metadata"]
  print(f"{r['id']}  {meta['path']:22s}  section={meta['section']}")

---

## 7. Chunk Identity and Versioning

How you assign ids determines whether incremental ingest works.

### 7.1 Id Strategies

| Strategy | Formula / rule | Pros | Cons |
|----------|----------------|------|------|
| **Content hash** | `hash(text)` | Auto-detect changes | Id changes on whitespace edits |
| **Path + index** | `f"{path}::{i}"` | Stable across minor edits | Must track index shifts on re-chunk |
| **Logical id** | `c2` (curriculum) | Human-readable updates | Manual mapping |
| **UUID per run** | `uuid4()` | Simple scripts | Orphans without delete pass |

### 7.2 Index Versioning

Version the **entire index artifact** when any of these change:

- Embedding model or `dimensions`
- Chunk size / overlap policy
- Metadata schema affecting filters

Example collection name: `notes_api_v3_embed_small_cosine`.

In [ ]:
def chunk_id_from_path(path: str, index: int) -> str:
    """Deterministic id: path + chunk index (common production pattern)."""
    return hashlib.sha256(f"{path}::{index}".encode()).hexdigest()[:16]


def content_hash_id(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()[:16]


sample_path = "docs/migrations.md"
print("path+index id:", chunk_id_from_path(sample_path, 0))
print("content hash id:", content_hash_id(INGEST_RECORDS[1]["text"]))
print("same text → same hash:", content_hash_id(INGEST_RECORDS[1]["text"]) == content_hash_id(INGEST_RECORDS[1]["text"]))

---

## 8. Embed and Upsert

The final ingest stages call your embedding API and vector store — same stack as notebook **02**.

### 8.1 Batching Embeddings

| Practice | Reason |
|----------|--------|
| Batch 50–200 texts per API call | Fewer round trips (**05.03**) |
| Record `usage.total_tokens` | Cost accounting |
| Validate `len(embedding)` constant | Catch model misconfiguration |

### 8.2 Upsert to Chroma

```python
collection.add(
    ids=[...],
    documents=[...],
    embeddings=[...],
    metadatas=[...],
)
```

Same `id` → overwrite. New `id` → insert. Missing delete pass → orphans when chunks removed.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(response.data, key=lambda item: item.index)
    return [item.embedding for item in ordered]


def fresh_collection(name: str = "kb_ingest_lesson"):
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name=name, metadata={"hnsw:space": "cosine"})


@dataclass
class IngestReport:
    chunks_written: int = 0
    tokens_embedded: int = 0
    errors: list[str] = field(default_factory=list)


def ingest_records(collection, records: list[dict], batch_size: int = 50) -> IngestReport:
    report = IngestReport()
    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        texts = [r["text"] for r in batch]
        if any(len(t.strip()) == 0 for t in texts):
            report.errors.append("empty chunk in batch")
            continue
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
        report.tokens_embedded += response.usage.total_tokens
        embeddings = [d.embedding for d in sorted(response.data, key=lambda x: x.index)]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=texts,
            embeddings=embeddings,
            metadatas=[r["metadata"] for r in batch],
        )
        report.chunks_written += len(batch)
    return report


col = fresh_collection()
report = ingest_records(col, INGEST_RECORDS)
print(f"Ingested {report.chunks_written} chunks, tokens={report.tokens_embedded}, errors={report.errors}")
print(f"Collection count: {col.count()}")

After ingest, `col` is what notebook **02**'s retriever searches. Keep **the same** `EMBED_MODEL` at query time.

---

## 9. Incremental vs Full Reindex

```
Full reindex:     drop collection ──► embed ALL ──► load
Incremental:      diff sources ──► embed CHANGED ──► upsert + delete stale ids
```

### 9.1 When to Use Each

| Corpus size | Typical approach |
|-------------|------------------|
| < 10k chunks | Nightly full rebuild is acceptable |
| 100k – 1M | Incremental + tombstone deletes |
| Multi-tenant | Per-tenant collections or partition keys |

### 9.2 Diff Workflow

1. Compute `new_ids` from current sources
2. Compare to `old_ids` in the index
3. **Added** → embed + upsert
4. **Removed** → `collection.delete(ids=...)`
5. **Unchanged** → skip (or re-embed if model version changed)

In [ ]:
def diff_ids(old_ids: set[str], new_ids: set[str]) -> dict[str, set[str]]:
    return {
        "added": new_ids - old_ids,
        "removed": old_ids - new_ids,
        "unchanged": old_ids & new_ids,
    }


old_index = {"c1", "c2", "c3", "c4"}
new_corpus = {"c1", "c2", "c5", "c4"}  # c3 removed, c5 added (autogenerate split)
delta = diff_ids(old_index, new_corpus)

for key, ids in delta.items():
    print(f"{key:10s}: {sorted(ids)}")

In [ ]:
# Illustrative: ingest cost vs corpus size (full reindex)
corpus_sizes = np.array([100, 1_000, 10_000, 100_000])
tokens_per_chunk = 120
cost_per_1m_tokens = 0.02  # illustrative USD for embed model
embed_cost = corpus_sizes * tokens_per_chunk / 1_000_000 * cost_per_1m_tokens

plt.figure(figsize=(7, 4))
plt.plot(corpus_sizes, embed_cost, marker="o", linewidth=2)
plt.xscale("log")
plt.xlabel("Chunks in corpus (log scale)")
plt.ylabel("Illustrative embed cost per full reindex (USD)")
plt.title("Plan incremental ingest as corpus grows")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 10. Data Quality Gates

Automate checks **before** vectors reach production.

| Check | Action if fail |
|-------|----------------|
| Empty chunk | Drop + log warning |
| Duplicate text | Deduplicate or merge metadata |
| Secret patterns (`sk-`, `password=`) | **Block ingest**, alert security |
| Embedding dimension mismatch | Abort batch |
| Language detection (optional) | Route to language-specific index |
| Max chunk token count | Re-chunk or truncate with log |

Failed gates should **fail the ingest job**, not silently skip — operators need visibility.

In [ ]:
SECRET_PATTERNS = [
    re.compile(r"sk-[A-Za-z0-9_-]{10,}"),
    re.compile(r"password\s*=\s*\S+", re.IGNORECASE),
]


def quality_gate(record: dict) -> tuple[bool, str]:
    text = record.get("text", "")
    if not text.strip():
        return False, "empty chunk"
    for pat in SECRET_PATTERNS:
        if pat.search(text):
            return False, "secret pattern detected"
    if "metadata" not in record or "source" not in record["metadata"]:
        return False, "missing source metadata"
    return True, "ok"


test_records = [
    INGEST_RECORDS[0],
    {"id": "bad", "text": "", "metadata": {"source": "x"}},
    {"id": "leak", "text": "apikey sk-proj-abc123secret", "metadata": {"source": "x"}},
]

for rec in test_records:
    ok, reason = quality_gate(rec)
    print(f"{rec['id']:6s}  pass={ok}  reason={reason}")

---

## 11. PII and Access Control

RAG can **leak** restricted text if every user searches the same index.

### 11.1 Defense Layers

| Control | Mechanism |
|---------|----------|
| **Ingest-time ACL tags** | `acl: hr-only` on metadata |
| **Query-time filter** | `where={"acl": user_clearance}` (**05.07**) |
| **Separate collections** | Per-tenant isolation |
| **Redaction** | Mask emails / SSN before embed |
| **Exclude at source** | Do not ingest highly sensitive dirs |

### 11.2 Principle

Embeddings are **not encryption**. Anyone with index access can approximate source text. Treat the vector store with the same ACL sensitivity as the original files.

In [ ]:
# Query-time ACL filter illustration (Chroma where clause)
USER_CLEARANCE = "internal"  # simulated logged-in user

allowed = [
    r for r in INGEST_RECORDS
    if r["metadata"].get("acl") == USER_CLEARANCE
]
print(f"User clearance={USER_CLEARANCE} may search {len(allowed)}/{len(INGEST_RECORDS)} chunks")
print("At query time: collection.query(..., where={\"acl\": \"internal\"})")

---

## 12. Observability for Ingest Jobs

Log structured metrics per ingest run:

| Metric | Why |
|--------|-----|
| `documents_processed` | Throughput |
| `chunks_created` / `deleted` | Index drift |
| `embed_tokens` | Cost |
| `duration_seconds` | SLA for nightly jobs |
| `error_count` | Loader/API failures |
| `index_version` | Reproducibility |

**Alert** when chunk count **drops sharply** — often a broken loader, expired repo token, or wrong path glob.

### 12.1 Sample Run Report

```python
{
  "run_id": "ingest-2026-03-01T02:00:00Z",
  "index_version": "notes_api_v3",
  "embed_model": "text-embedding-3-small",
  "chunks_written": 5,
  "chunks_deleted": 1,
  "tokens_embedded": 412,
  "duration_s": 3.2,
  "status": "success"
}
```

In [ ]:
run_report = {
    "run_id": datetime.now(timezone.utc).strftime("ingest-%Y-%m-%dT%H:%M:%SZ"),
    "index_version": "notes_api_v3",
    "embed_model": EMBED_MODEL,
    "chunks_written": report.chunks_written,
    "tokens_embedded": report.tokens_embedded,
    "status": "success" if not report.errors else "partial",
}
for k, v in run_report.items():
    print(f"{k:18s}: {v}")

---

## 13. CI/CD Integration

Treat ingest like a build step:

```
git push main ──► CI ──► run ingest job ──► deploy new index pointer ──► RAG API reads new collection
```

| Practice | Benefit |
|----------|--------|
| Ingest on doc changes only | Saves embed cost |
| Blue/green index names | Roll back by swapping pointer |
| Store `index_version` in API config | Trace which KB answered a query |
| Block deploy if quality gates fail | No poisoned index in prod |

---

## 14. Connecting Ingest to the Online RAG Path

Ingest output must match what notebook **02** expects:

```python
collection.add(ids=..., documents=..., embeddings=..., metadatas=...)
# Online:
hits = collection.query(query_embeddings=[q_vec], n_results=k, include=[...])
```

| Rule | Reason |
|------|--------|
| Same `EMBED_MODEL` at ingest and query | Comparable similarity |
| Same `hnsw:space` metric | Consistent distance |
| Metadata fields query filters use | Filters won't work if not ingested |

When you change embedding model: **full reindex** + update API config — never mix vectors from two models in one collection.

---

## 15. Multi-Tenant and Rollback

### 15.1 Multi-Tenant Patterns

| Pattern | Isolation |
|---------|----------|
| **Collection per tenant** | Strongest; more ops overhead |
| **Shared collection + `tenant_id` filter** | Fewer collections; strict filter discipline |
| **Separate embed indexes per region** | Data residency |

### 15.2 Rollback

Keep the previous index version for N days. If a bad ingest ships, point the API to `notes_api_v2` while fixing `v3`. Vector data is **immutable per version** — rollback is a config change, not re-embedding.

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Ingest secrets | Permanent leak risk | Secret scan gates |
| Random UUID ids each run | Orphan vectors accumulate | Stable ids + delete diff |
| No `updated_at` metadata | Cannot detect staleness | Timestamp on every chunk |
| Different embed model at query | Silent retrieval degradation | Versioned reindex |
| Skip delete on removed chunks | Wrong outdated answers | Incremental diff |
| One giant PDF chunk | Vague retrieval | **05.04** chunking |
| No ingest logs | Blind to index shrink | Structured run reports |

---

## 17. Debugging Ingest Failures

1. **Source glob** — Did the job find the expected file count?
2. **Loader errors** — Any parse exceptions per path?
3. **Chunk count** — Order-of-magnitude match with prior run?
4. **Embed API** — Rate limits, auth, dimension errors?
5. **Upsert** — `collection.count()` after job?
6. **Sample query** — Manual `query` for a known phrase (**05.09** eval set)?

If ingest succeeds but RAG fails, switch to the retrieval-vs-generation checklist in notebook **02** §16.

---

## 18. Summary

A RAG knowledge base is a **managed artifact**: scoped sources, clean loaders, stable chunk ids, rich metadata, batched embeddings, and idempotent upserts. Ingest runs offline; query runs online — keep embedding models aligned across both.

Incremental diff (`added` / `removed` / `unchanged`), quality gates, ACL metadata, and ingest observability prevent silent index corruption. Chunking depth lives in **05.04**; this notebook closes the gap between **raw docs** and the index that **02** retrieves from.

Demonstrations built the full **c1–c5** corpus with metadata, deterministic id helpers, `ingest_records()`, diff logic, quality gates, and a sample run report.

Back: **03. RAG Architecture Patterns**. Next: **05. Prompting and Context Injection for RAG** — how retrieved text becomes effective LLM input.